# 영상 요약과 이벤트 탐지

1946년 가정학(home economics) 교육 영상 `Arranging a Buffet Supper`를 사용해
**영상 요약**과 **타임스탬프가 포함된 이벤트 탐지(JSON)**를 실습합니다.

## 사전 준비
```bash
uv sync
ollama pull qwen3-vl:4b
```

In [ ]:
import video_utils as vu

VIDEO = "data/2_cooking_steps_buffet_supper_1946.mp4"
info = vu.get_video_info(VIDEO)
info

## 1. 영상 요약 (Video Summarization)
균등 샘플링한 프레임 전체를 한 번에 전달해 전체 흐름을 요약합니다. 영상이
6분(360초)이므로 8장만 뽑아도 45초 간격 — 큰 그림을 파악하기엔 충분합니다.

In [ ]:
summary_frames = vu.extract_frames_uniform(VIDEO, n_frames=8)
print("타임스탬프(초):", [round(t, 1) for t, _ in summary_frames])
vu.show_frames(summary_frames)

In [ ]:
summary = vu.ask_video(
    summary_frames,
    "아래 프레임들은 상차림(뷔페 준비) 교육 영상 전체를 균등한 간격으로 "
    "추출한 것이야. 이 영상에서 어떤 준비 과정이 시간 순서대로 진행되는지 "
    "3~4문장으로 요약해줘. 프레임에서 확인되지 않는 내용은 추측하지 마.",
)
print(summary)

## 2. 이벤트 탐지 (Event Detection)
"장면/작업이 바뀌는 시점"을 찾아 구조화된 JSON으로 뽑습니다. 프롬프트에
타임스탬프가 함께 전달되므로(`ask_video`가 자동으로 붙여줌), 모델이 답변에
실제 초 단위 값을 채워 넣을 수 있습니다.

In [ ]:
event_frames = vu.extract_frames_uniform(VIDEO, n_frames=10)
print("타임스탬프(초):", [round(t, 1) for t, _ in event_frames])
vu.show_frames(event_frames)

In [ ]:
schema_prompt = """
아래 프레임들에서 준비 작업이 바뀌는 지점을 찾아서, 다음 JSON 스키마로만 답해.
다른 설명 문장은 출력하지 마.
{
  "events": [
    {"timestamp_sec": number, "event": "string", "description": "string"}
  ]
}
확인되지 않으면 빈 리스트를 반환해.
"""
events = vu.ask_video_json(event_frames, schema_prompt)
events

## 3. 결과 표로 정리
JSON 그대로도 후속 처리(DB 저장 등)가 가능하지만, 검수 목적으로 표로도
확인해봅니다.

In [ ]:
for e in events.get("events", []):
    print(f"{e['timestamp_sec']:>5}s | {e['event']:<20} | {e['description']}")

## 한계 확인
샘플링 간격(약 36초)보다 짧게 스쳐 지나가는 동작은 애초에 프레임에 담기지
않으므로 탐지될 수 없습니다. 더 정밀한 이벤트 탐지가 필요하면
`extract_frames_fixed_fps`로 더 촘촘히 샘플링하거나, 의심 구간만
`extract_frames_scene_change`로 다시 뽑아 확인하는 2단계 접근이 효과적입니다.